## 1. Generative Model: LoRA-fine-tuned Phi-2 on Dolly test set

#### 1.1 Load Prereqs and Base Model

In [2]:
import torch
from transformers import AutoTokenizer, PreTrainedTokenizerBase, AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

from typing import cast
import numpy as np

In [3]:
model_name = "microsoft/phi-2"

# quantization via bitsandbytes custom kernels for consumer GPU: save memory, faster
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = cast(
    PreTrainedTokenizerBase,
    cast(object, AutoTokenizer.from_pretrained(model_name))
)

# Phi-2 doesn't have a pad token; set to eos token for safer gradient contribution
tokenizer.pad_token = tokenizer.eos_token

In [4]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",      # fit layers on GPUs before offloading to CPU RAM
    trust_remote_code=True, # well-known model
)

Loading weights: 100%|██████████| 453/453 [00:08<00:00, 51.14it/s]


In [ ]:
base_model.gradient_checkpointing_enable() # for training: recomputes activation during the backward pass

#### 1.2 Load Dataset

In [18]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
dataset

Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})

In [19]:
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
val_dataset   = split_dataset["test"]

In [20]:
def format_instruction(example):
    instruction = f"### Instruction: \n{example['instruction']}\n\n"
    context = f"### Context: \n{example['context']}\n\n"
    response = f'### Response: \n{example["response"]}\n\n'

    return {"text": instruction + context + response}

formatted_train = train_dataset.map(format_instruction)
formatted_val   = val_dataset.map(format_instruction)

In [12]:
print(formatted_train[1000]["text"])

### Instruction: 
What are close cities to Legonice?

### Context: 
Łęgonice [wɛnɡɔˈnit͡sɛ] is a village in the administrative district of Gmina Nowe Miasto nad Pilicą, within Grójec County, Masovian Voivodeship, in east-central Poland. It lies approximately 3 kilometres (2 mi) west of Nowe Miasto nad Pilicą, 36 km (22 mi) south-west of Grójec, and 74 km (46 mi) south-west of Warsaw.

The village has a population of 440.

### Response: 
Close cities to Legonice are Nowe Miasto nad Pilica, Grojec, and Warsaw.




In [38]:
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

tokenized_train = formatted_train.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_train.column_names
)

tokenized_val = formatted_val.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_val.column_names
)

#### 1.3 Configure and Apply LoRA

In [14]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    task_type=TaskType.CAUSAL_LM,
    lora_dropout=0.05, # slightly lower for generation
    bias="none",
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 2,621,440 || all params: 2,782,305,280 || trainable%: 0.0942


#### 1.4 Training

In [2]:
import torch
torch.cuda.is_bf16_supported()

True

In [17]:
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU memory allocated: 1.83 GB
GPU memory reserved: 2.04 GB


In [13]:
# error-handling
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [15]:
from transformers import TrainerCallback
import gc

class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 100 == 0:
            gc.collect()
            torch.cuda.empty_cache()

In [16]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="D:/models/phi2-dolly-lora",
    per_device_train_batch_size=1, # higher param model -> decrease to fit VRAM
    gradient_accumulation_steps=8, # for an effective batch size of 2 x 4 = 8
    num_train_epochs=3,
    learning_rate=2e-4, # larger dataset -> increase
    optim="adamw_8bit", # memory
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=False,
    logging_steps=50,   # expecting slower training
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    report_to="none",   # exploratory
    max_grad_norm=1.0,  # prevent explosive clipping
    lr_scheduler_type="cosine", # more stable learning rate
    warmup_steps=0.3,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model = peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

trainer.add_callback(ClearCacheCallback())

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.837604,1.777934
2,1.775671,1.744360
3,1.756433,1.735749


TrainOutput(global_step=4503, training_loss=1.8477703371287293, metrics={'train_runtime': 36809.0314, 'train_samples_per_second': 0.979, 'train_steps_per_second': 0.122, 'total_flos': 9.698358216563712e+16, 'train_loss': 1.8477703371287293, 'epoch': 3.0})

In [18]:
peft_model.save_pretrained("D:/models/phi2-dolly-lora-adapter")
tokenizer.save_pretrained("D:/models/phi2-dolly-lora-adapter")

('D:/phi2-dolly-lora-adapter\\tokenizer_config.json',
 'D:/phi2-dolly-lora-adapter\\tokenizer.json')

#### 1.5 Evaluation

In [5]:
adapter_path = "D:/models/phi2-dolly-lora-adapter"
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PhiForCausalLM(
      (model): PhiModel(
        (embed_tokens): Embedding(51200, 2560)
        (layers): ModuleList(
          (0-31): 32 x PhiDecoderLayer(
            (self_attn): PhiAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=2560, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2560, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear4bit(in_f

In [14]:
# make sure model is on GPU instead of CPU
print(f"wrapper device (base model): {model.device}")
print(f"first-parameter device (weights): {next(model.base_model.parameters()).device}")

Wrapper device (base model): cuda:0
First-parameter device (weights): cuda:0


In [22]:
# sample generation

prompt = "### Instruction:\nexplain what LoRA is in simple terms.\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7, # introduce randomness for logits outputs
        do_sample=True,  # prevent greedy decoding of highest probability
        top_p=0.9,       # only consider tokens covering 90% of prob. mass
        pad_token_id=tokenizer.eos_token_id,
        stop_strings=['###', "\n\n###"],    # prevent extra metadata i.e. `Context`
        tokenizer=tokenizer                 # needed for stop_strings
    )
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = generated_text.split('### Response:\n')[-1].strip()
print(response)

LoRa is a technology that is used for communication. LoRa is very low power and is used to send and receive information. It is used in many different applications such as home automation, smart cities, and agriculture. LoRa uses radio waves to send and receive information.


In [24]:
# generate for a subset of the test set for evaluation

from tqdm import tqdm

generated = []
references = []

for ex in tqdm(formatted_val.select(range(100)), desc="Generating..."):
    full_text = ex["text"]
    prompt = full_text.split("###Response:")[0] + "### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            stop_strings=["###", '\n\n###'], #
            tokenizer=tokenizer
        )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = generated_text.split('### Response:\n')[-1].strip()

    generated.append(response)
    references.append(ex['response'])

Generating...: 100%|██████████| 100/100 [15:22<00:00,  9.22s/it]


In [37]:
import evaluate

rogue = evaluate.load("rouge")
bleu = evaluate.load("bleu")
bert = evaluate.load("bertscore")

rouge_results = rogue.compute(predictions=generated, references=references)
print("ROUGE scores:", rouge_results)

bleu_results = bleu.compute(predictions=generated, references=[[ref] for ref in references])
print("BLEU score:", bleu_results)

bert_results = bert.compute(predictions=generated, references=references, lang="en")
print("BERTScore (F1):", sum(bert_results["f1"])/len(bert_results["f1"]))

ROUGE scores: {'rouge1': np.float64(0.6203908006661714), 'rouge2': np.float64(0.5261943997621163), 'rougeL': np.float64(0.5728374875550256), 'rougeLsum': np.float64(0.5808499389195605)}
BLEU score: {'bleu': 0.3165237974191929, 'precisions': [0.40716748511127365, 0.31601731601731603, 0.2863088250987088, 0.2724625175277748], 'brevity_penalty': 1.0, 'length_ratio': 1.612636899747262, 'translation_length': 9571, 'reference_length': 5935}


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 5577.93it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1): 0.9148839730024337


In [42]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
test_loader = DataLoader(tokenized_val.select(range(500)), batch_size=4, collate_fn=data_collator)

total_loss   = 0
total_tokens = 0
with torch.no_grad():
    for batch in tqdm(test_loader):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item() * batch['input_ids'].numel()
        total_tokens += batch['input_ids'].numel()

perplexity = torch.exp(torch.tensor(total_loss / total_tokens))
print(f"perplexity: {perplexity}")

100%|██████████| 125/125 [01:18<00:00,  1.60it/s]

perplexity: 6.108673572540283


**Notes**
* Training loss (1.85 → 1.75) decreasing suggests the model is learning and not overfitting dramatically. Validation loss (1.777 → 1.744 → 1.735) suggests the model approaching convergence.
* Perplexity (exp(loss)) measures the model's predictive uncertainty -- a score of 6.1 is moderate for a 2.7B parameter model. However, because the Dolly validation set is structurally and thematically similar to Phi-2's pre-training data, this score likely reflects in-distribution familiarity rather than true generalization. The model may perform worse on out-of-distribution prompts. Nonetheless, the downward trend in loss confirms that fine-tuning with LoRA successfully adapted the model to the instruction-following format, leaving room for further improvement with more data or hyperparameter tuning.
* ROUGE measures overlap between generated text and reference responses. It is recall-oriented. The sub‑metrics (unigram 0.620, bigram 0.526, longest common subsequence 0.573) are respectable, especially because ROUGE‑L stays close to ROUGE‑1, indicating coherent sentence‑level structure.
* BLEU  measures precision of n-gram overlap. The overall score (0.317) is moderate. The length ratio (1.61) tells us that the outputs are longer, more verbose, than references, and the per‑n‑gram precisions (0.41, 0.32, 0.29, 0.27) show reasonable overlap, especially at 1‑gram and 2‑gram levels. For instruction following, lower `temperature` could encourage conciseness.
* BERTScore uses contextual embeddings to measure semantic similarity. Our F1 score (0.915) means generated responses are semantically very close to the references, even when wording differs. This is the strongest evidence that fine‑tuning succeeded.

Overall, without a baseline, we can still infer that the model has learned to follow instructions and generate relevant responses. Hyperparameter tuning (e.g., LoRA capacity, more epochs) or changing datasets (larger or diversify) could improve verbosity and ROUGE, but for learning the RAG pipeline, this model is sufficient.

## 2. Retriever

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Tuple, Dict, Set
import numpy as np
import pandas as pd
import ir_datasets

In [3]:
_dataset = ir_datasets.load("beir/nfcorpus/dev")
dataset

Dataset(id='beir/nfcorpus/dev', provides=['docs', 'queries', 'qrels'])

In [3]:
docs_df = pd.DataFrame(
    {
        "doc_id": d.doc_id,
        "title": d.title,
        "text": d.text,
        "url": d.url
    }
    for d in dataset.docs_iter()
)
print(docs_df.shape)
#docs_df.head()

(3633, 4)


In [4]:
queries_df = pd.DataFrame(
    {
        "query_id": q.query_id,
        "text": q.text,
        "url": q.url,
    }
    for q in dataset.queries_iter()
)
print(queries_df.shape)
#queries_df.head()

(324, 3)


In [5]:
qrels_df = pd.DataFrame(
    {
        "query_id": r.query_id,
        "doc_id": r.doc_id,
        "relevance": r.relevance,
        "iteration": r.iteration,
    }
    for r in dataset.qrels_iter()
)
print(qrels_df.shape)
#qrels_df.head()

(11385, 4)


In [6]:
# create dictionaries of relevant information
corpus = {
    row.doc_id: f"{row.title or ''} {row.text or ''}".strip()
    for row in docs_df.itertuples(index=False)
}

queries = {
    row.query_id: row.text
    for row in queries_df.itertuples(index=False)
}

relevant_docs = (
    qrels_df[qrels_df['relevance'] > 0]
    .groupby('query_id')['doc_id']
    .apply(set)
    .to_dict()
)

In [39]:
# average document length
lengths = np.array([len(text.split()) for text in corpus.values()])

print(f"Average words/doc: {lengths.mean():.2f}")
print(f"Median words/doc: {np.median(lengths):.2f}")
print(f"25th percentile: {np.percentile(lengths, 25):.2f}")
print(f"75th percentile: {np.percentile(lengths, 75):.2f}")
print(f"Min: {lengths.min()}")
print(f"Max: {lengths.max()}")

Average words/doc: 233.77
Median words/doc: 237.00
25th percentile: 185.00
75th percentile: 273.00
Min: 17
Max: 1481


**Note:** Document length is moderate at most--doing so ended up hurting results--so chunking will be saved for our final retriever.

In [29]:
import re
import unicodedata

from nltk.stem import PorterStemmer

# performs better than lemmatizing (too aggressive for biomedical retrieval)
stemmer = PorterStemmer()

GREEK_MAP = {
    "α": "alpha",
    "β": "beta",
    "γ": "gamma",
    "δ": "delta",
    "κ": "kappa",
    "μ": "mu",
}

def normalize_text(text):
    if text is None:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = (text.lower()
        .replace("-", " ")
        .replace("/", " ")
        .replace("&", " and ")
    )

    # remove punctuation but keep letters/numbers/spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    # collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    for symbol, name in GREEK_MAP.items():
        text = text.replace(symbol, f" {name} ")

    return text

In [ ]:
def tokenize(text, use_stemming=False):
    text = normalize_text(text)
    tokens = text.split()

    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]

    return tokens

In [ ]:
doc_ids = list(corpus.keys())
doc_texts = [corpus[doc_id] for doc_id in doc_ids]
tokenized_docs = [tokenize(text, use_stemming=True) for text in doc_texts]

## 3. Generator-Retriever Pairing/Conditioning

## 4. Geometry Analysis